In [63]:
!apt-get update -qq
!apt-get install -y mysql-server -qq
!pip install -q pandas sqlalchemy pymysql


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [64]:
!service mysql start

[ OK ]


In [65]:
from google.colab import files

uploaded = files.upload()

Saving 12923129-Advanced_SQL_Data_set_for_MP_4 (1).zip to 12923129-Advanced_SQL_Data_set_for_MP_4 (1) (3).zip


In [66]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("Extracted files:")
print(os.listdir('/content'))

Extracted files:
['.config', '12923129-Advanced_SQL_Data_set_for_MP_4 (1).zip', '12923129-Advanced_SQL_Data_set_for_MP_4 (1) (2).zip', '12923129-Advanced_SQL_Data_set_for_MP_4.zip', '12923129-Advanced_SQL_Data_set_for_MP_4 (1) (1).zip', 'Advanced SQL Data set for MP 4.sql', '12923129-Advanced_SQL_Data_set_for_MP_4 (2).zip', '12923129-Advanced_SQL_Data_set_for_MP_4 (1) (3).zip', 'sample_data']


In [67]:
import os

sql_files = []

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.lower().endswith('.sql'):
            sql_files.append(os.path.join(root, file))

print("SQL files found:")
for f in sql_files:
    print(f)

SQL files found:
/content/Advanced SQL Data set for MP 4.sql


In [68]:
import subprocess

sql_file = sql_files[0]

print("Loading:", sql_file)

with open(sql_file, "rb") as f:
    result = subprocess.run(
        ["mysql"],
        stdin=f,
        capture_output=True,
        text=True
    )

print("Return code:", result.returncode)

if result.stderr:
    print(result.stderr)
else:
    print("Database loaded successfully!")

Loading: /content/Advanced SQL Data set for MP 4.sql
Return code: 0
Database loaded successfully!


In [69]:
!mysql -e "SHOW DATABASES;"

+--------------------+
| Database           |
+--------------------+
| bingeplay          |
| information_schema |
| mysql              |
| performance_schema |
| sys                |
+--------------------+


In [70]:
!mysql -e "USE bingeplay; SHOW TABLES;"

+---------------------+
| Tables_in_bingeplay |
+---------------------+
| ratings             |
| shows               |
| subscriptions       |
| users               |
| watch_sessions      |
+---------------------+


In [71]:
!mysql -e "USE bingeplay; SELECT 'users' AS tbl, COUNT(*) AS row_count FROM users;"

+-------+-----------+
| tbl   | row_count |
+-------+-----------+
| users |      3000 |
+-------+-----------+


In [72]:
!mysql -e "USE bingeplay; SELECT 'subscriptions' AS tbl, COUNT(*) AS row_count FROM subscriptions;"

+---------------+-----------+
| tbl           | row_count |
+---------------+-----------+
| subscriptions |      4497 |
+---------------+-----------+


In [73]:
!mysql -e "USE bingeplay; SELECT 'shows' AS tbl, COUNT(*) AS row_count FROM shows;"

+-------+-----------+
| tbl   | row_count |
+-------+-----------+
| shows |       100 |
+-------+-----------+


In [74]:
!mysql -e "USE bingeplay; SELECT 'watch_sessions' AS tbl, COUNT(*) AS row_count FROM watch_sessions;"

+----------------+-----------+
| tbl            | row_count |
+----------------+-----------+
| watch_sessions |    100351 |
+----------------+-----------+


In [75]:
!mysql -e "USE bingeplay; SELECT 'ratings' AS tbl, COUNT(*) AS row_count FROM ratings;"

+---------+-----------+
| tbl     | row_count |
+---------+-----------+
| ratings |      5000 |
+---------+-----------+


In [76]:
import pymysql
import pandas as pd
from sqlalchemy import create_engine

conn = pymysql.connect(
    unix_socket="/var/run/mysqld/mysqld.sock",
    user="root",
    password="",
    database="bingeplay"
)

print("MySQL connected!")

conn.close()


MySQL connected!


In [77]:
from sqlalchemy import create_engine

engine = create_engine(
    "mysql+pymysql://root:@localhost/bingeplay"
)

print("SQLAlchemy connection ready!")

SQLAlchemy connection ready!


**ANSWERS**

## Q1 — Active Revenue

**Active subscriptions: 2,340**

**Total monthly recurring revenue: ₹784,260**

As of 30 June 2024, BingePlay has 2,340 active subscriptions generating ₹784,260 in monthly recurring revenue.

In [78]:
from sqlalchemy import create_engine

engine = create_engine(
    "mysql+pymysql://root:@localhost/bingeplay",
    connect_args={"unix_socket": "/var/run/mysqld/mysqld.sock"}
)

print("SQLAlchemy connection ready!")

SQLAlchemy connection ready!


In [79]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 active_subscriptions  monthly_revenue_inr
                 2340             784260.0


## Q2 — Signup Momentum
**January:** 350 signups  
**February:** 400 signups  
**March:** 500 signups  
**April:** 550 signups  
**May:** 600 signups  
**June:** 600 signups

**Highest signups:** May and June, tied at 600 signups each.

In [80]:
query = """
SELECT
    MONTH(signup_date) AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date)
ORDER BY month;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

highest = result.loc[result['signup_count'].idxmax()]

print("\nMonth with highest signups:")
print(f"{highest['month']} ({highest['signup_count']} signups)")

 month  signup_count
     1           350
     2           400
     3           500
     4           550
     5           600
     6           600

Month with highest signups:
5 (600 signups)


## Q3 — Device Analytics

The device analytics results are:

- Laptop: 15,105 sessions, 453,434 watch minutes, 30.02 average minutes, 60.51% completion
- Mobile: 50,172 sessions, 1,504,355 watch minutes, 29.98 average minutes, 60.24% completion
- Tablet: 7,091 sessions, 210,733 watch minutes, 29.72 average minutes, 59.79% completion
- TV: 27,981 sessions, 840,595 watch minutes, 30.04 average minutes, 59.98% completion

NULL user_id sessions were excluded.

In [81]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*), 2)
        AS completion_rate_percent
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

device_type  total_sessions  total_watch_minutes  avg_watch_minutes  completion_rate_percent
     Laptop           15105             453434.0              30.02                    60.51
     Mobile           50172            1504355.0              29.98                    60.24
     Tablet            7091             210733.0              29.72                    59.79
         TV           27981             840595.0              30.04                    59.98


## Q4 — Rating Distribution

**1 star:** 234 ratings (4.68%)  
**2 stars:** 352 ratings (7.04%)  
**3 stars:** 847 ratings (16.94%)  
**4 stars:** 1,781 ratings (35.62%)  
**5 stars:** 1,786 ratings (35.72%)

**Percentage of ratings that are 4 or 5 stars: 71.34%**

In [82]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

high_ratings = pd.read_sql("""
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentage_4_or_5_stars
FROM ratings;
""", engine)

print("\nPercentage of ratings that are 4 or 5 stars:")
print(high_ratings.to_string(index=False))

 stars  rating_count  percentage
     1           234        4.68
     2           352        7.04
     3           847       16.94
     4          1781       35.62
     5          1786       35.72

Percentage of ratings that are 4 or 5 stars:
 percentage_4_or_5_stars
                   71.34


## Q5 — Originals vs Acquired

**BingePlay Originals:** 30 shows, average IMDb rating 7.92, average release year 2020.37

**Acquired Content:** 70 shows, average IMDb rating 6.63, average release year 2020.73

BingePlay Originals have an average IMDb rating of 7.92 compared with 6.63 for acquired content, a difference of **1.29 rating points**.

In [83]:
query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Originals'
        ELSE 'Acquired Content'
    END AS content_type,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

       content_type  number_of_shows  avg_imdb_rating  avg_release_year
BingePlay Originals               30             7.92           2020.37
   Acquired Content               70             6.63           2020.73


## Q6 — Binge Day Detection

**Total binge days in Q2 2024: 414**

**User with the most binge days: U02956**

**Binge days for U02956: 8**

A binge day is counted when the same user watches the same show at least 5 times on the same calendar date.

In [84]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date
    FROM watch_sessions
    WHERE session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),

user_binge_counts AS (
    SELECT
        user_id,
        COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
)

SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    user_id AS top_user_id,
    binge_days AS top_user_binge_days
FROM user_binge_counts
ORDER BY binge_days DESC, user_id
LIMIT 1;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 total_binge_days top_user_id  top_user_binge_days
              414      U02956                    8


## Q7 — Q1 Signups Who Never Watched

**Total Q1 2024 signups: 1,250**

**Q1 signups who never watched anything: 226**

A NULL-safe LEFT JOIN was used to correctly identify users with no watch sessions.

In [85]:
query = """
SELECT
    COUNT(DISTINCT u.user_id) AS total_q1_signups,
    COUNT(DISTINCT CASE
        WHEN ws.session_id IS NULL THEN u.user_id
    END) AS never_watched
FROM users u
LEFT JOIN watch_sessions ws
    ON u.user_id = ws.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 total_q1_signups  never_watched
             1250            226


## Q8 — Over-paying Premium/Family Users

**Over-paying Premium/Family users: 212**

These are users on Premium or Family plans whose watch history contains only shows available on the Basic plan.

In [86]:
query = """
WITH current_plans AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)

SELECT COUNT(*) AS overpaying_users
FROM current_plans cp
WHERE cp.rn = 1
  AND cp.plan IN ('Premium', 'Family')
  AND EXISTS (
      SELECT 1
      FROM watch_sessions ws
      WHERE ws.user_id = cp.user_id
  )
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s
        ON ws.show_id = s.show_id
      WHERE ws.user_id = cp.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 overpaying_users
                7


## Q9 — Upgrade Success Cohort

**Qualifying users: 55**

**Average days from signup to first upgrade: 64.96 days**

These users signed up in January 2024, started with Basic, later upgraded to Premium or Family, and were still active as of 30 June 2024.

In [87]:
query = """
WITH ordered_subscriptions AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),

first_basic AS (
    SELECT
        user_id,
        start_date AS basic_start_date
    FROM ordered_subscriptions
    WHERE rn = 1
      AND plan = 'Basic'
),

first_upgrade AS (
    SELECT
        s.user_id,
        MIN(s.start_date) AS first_upgrade_date
    FROM subscriptions s
    JOIN first_basic fb
        ON s.user_id = fb.user_id
    WHERE s.plan IN ('Premium', 'Family')
      AND s.start_date > fb.basic_start_date
    GROUP BY s.user_id
),

still_active AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)

SELECT
    COUNT(*) AS qualifying_users,
    ROUND(
        AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)),
        2
    ) AS avg_days_to_first_upgrade
FROM users u
JOIN first_basic fb
    ON u.user_id = fb.user_id
JOIN first_upgrade fu
    ON u.user_id = fu.user_id
JOIN still_active sa
    ON u.user_id = sa.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01';
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 qualifying_users  avg_days_to_first_upgrade
               55                      64.96


## Q10 — Cliffhanger Comebacks

**Total cliffhanger comeback events: 4,345**

**Show with the most comeback events:** S088 — Rayalaseema Raga

**Comeback events for S088: 64**

A comeback is counted when a user has an incomplete session and watches the same show again within 1–7 days.

In [88]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        w1.user_id,
        w1.show_id,
        w1.session_date AS incomplete_date
    FROM watch_sessions w1
    JOIN watch_sessions w2
        ON w1.user_id = w2.user_id
        AND w1.show_id = w2.show_id
        AND w2.session_date BETWEEN
            DATE_ADD(w1.session_date, INTERVAL 1 DAY)
            AND DATE_ADD(w1.session_date, INTERVAL 7 DAY)
    WHERE w1.completed = 0
      AND w1.user_id IS NOT NULL
),

show_counts AS (
    SELECT
        show_id,
        COUNT(*) AS comeback_count
    FROM comeback_events
    GROUP BY show_id
),

top_show AS (
    SELECT
        show_id,
        comeback_count
    FROM show_counts
    ORDER BY comeback_count DESC, show_id
    LIMIT 1
)

SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    ts.show_id,
    s.title,
    ts.comeback_count
FROM top_show ts
JOIN shows s
    ON ts.show_id = s.show_id;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 total_comeback_events show_id            title  comeback_count
                  4345    S088 Rayalaseema Raga              64


## Q11 — Consecutive-Week Engagement

**Users with a 4+ consecutive-week streak: 1,675**

**Longest streak: 26 weeks**

**User with the longest streak: U00213**

ISO week-based consecutive viewing activity was used to identify the streaks.

In [89]:
query = """
WITH weekly_activity AS (
    SELECT DISTINCT
        user_id,
        DATE_SUB(session_date, INTERVAL WEEKDAY(session_date) DAY) AS week_start
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),

numbered_weeks AS (
    SELECT
        user_id,
        week_start,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_start
        ) AS rn
    FROM weekly_activity
),

grouped_streaks AS (
    SELECT
        user_id,
        week_start,
        DATEDIFF(week_start, '2024-01-01') / 7 - rn AS grp
    FROM numbered_weeks
),

streak_lengths AS (
    SELECT
        user_id,
        grp,
        COUNT(*) AS streak_length
    FROM grouped_streaks
    GROUP BY user_id, grp
),

qualifying_users AS (
    SELECT DISTINCT user_id
    FROM streak_lengths
    WHERE streak_length >= 4
),

longest AS (
    SELECT
        user_id,
        MAX(streak_length) AS longest_streak
    FROM streak_lengths
    GROUP BY user_id
)

SELECT
    (SELECT COUNT(*) FROM qualifying_users) AS users_with_4plus_week_streak,
    (SELECT MAX(longest_streak) FROM longest) AS longest_streak_weeks,
    (
        SELECT user_id
        FROM longest
        ORDER BY longest_streak DESC, user_id
        LIMIT 1
    ) AS user_with_longest_streak;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

 users_with_4plus_week_streak  longest_streak_weeks user_with_longest_streak
                         1675                    26                   U00213


## Q12 — Churn Signal Detection

**Total churn-signal users: 521**

These users experienced a drop of 50% or more in total watch minutes in June 2024 compared with May 2024. Only users with watch activity in May were considered.

In [90]:
query = """
WITH monthly_watch AS (
    SELECT
        u.user_id,
        u.name,
        SUM(
            CASE
                WHEN ws.session_date >= '2024-05-01'
                 AND ws.session_date < '2024-06-01'
                THEN ws.watch_minutes
                ELSE 0
            END
        ) AS may_watch_minutes,
        SUM(
            CASE
                WHEN ws.session_date >= '2024-06-01'
                 AND ws.session_date < '2024-07-01'
                THEN ws.watch_minutes
                ELSE 0
            END
        ) AS june_watch_minutes
    FROM users u
    JOIN watch_sessions ws
        ON u.user_id = ws.user_id
    WHERE ws.session_date >= '2024-05-01'
      AND ws.session_date < '2024-07-01'
    GROUP BY u.user_id, u.name
),

churn_signals AS (
    SELECT
        user_id,
        name,
        may_watch_minutes,
        june_watch_minutes,
        ROUND(
            100.0 * (may_watch_minutes - june_watch_minutes)
            / may_watch_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch
    WHERE may_watch_minutes > 0
      AND june_watch_minutes <= may_watch_minutes * 0.5
)

SELECT
    user_id,
    name,
    may_watch_minutes,
    june_watch_minutes,
    drop_percentage,
    COUNT(*) OVER () AS total_churn_signal_users
FROM churn_signals
ORDER BY user_id;
"""

result = pd.read_sql(query, engine)

print(result.to_string(index=False))

user_id                name  may_watch_minutes  june_watch_minutes  drop_percentage  total_churn_signal_users
 U00021     Sneha Mukherjee             3104.0               456.0            85.31                       521
 U00023      Amit Mukherjee               43.0                 0.0           100.00                       521
 U00024          Rohit Bhat              757.0                66.0            91.28                       521
 U00034         Rekha Reddy               94.0                43.0            54.26                       521
 U00046        Meera Pillai             2355.0               366.0            84.46                       521
 U00067          Meena Bose              131.0                34.0            74.05                       521
 U00077        Tara Agarwal              429.0                64.0            85.08                       521
 U00083         Aisha Verma              570.0                65.0            88.60                       521
 U00084   